# Lab 11 — Semantic Search with Embeddings

This notebook adds semantic search capabilities to the music artist pipeline built in Lab 10.  
Instead of matching exact words, we use **sentence embeddings** to find artists by *meaning*.  

**Dataset:** Last.fm music artists (name, listeners, playcount, source collection)  
**Model:** `all-MiniLM-L6-v2` — 384-dimensional sentence embeddings  
**Vector DB:** ChromaDB (persistent, cosine similarity)  

---
## Contents
1. Install & imports
2. Understanding embeddings
3. Cosine similarity, dot product, Euclidean distance
4. ChromaDB setup
5. Populating ChromaDB with artist data
6. Querying ChromaDB
7. Metadata filtering
8. Search engine module demo
9. Complete semantic search system
10. Keyword vs semantic comparison
11. Hybrid search (RRF)
12. Pipeline integration
13. Analytical questions

---
## Part 1 — Install Required Libraries

In [ ]:
# Uncomment and run once if the packages are not yet installed
# !pip install "sentence-transformers>=2.7.0" "chromadb>=0.5.0" "torch>=2.0.0"

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# Add src/ to the path so we can import our modules
ROOT = Path(os.getcwd()).parent
SRC  = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('Root:', ROOT)
print('Python path includes src:', str(SRC) in sys.path)

---
## Part 2 — Understanding Embeddings

An **embedding** is a list of floating-point numbers that represents the *meaning* of a text.  
The `all-MiniLM-L6-v2` model always produces **384 numbers** per text, no matter the length.

**Key property:** texts with similar meanings have embedding vectors that are *close together* in 384-dimensional space.

In [ ]:
from embeddings.embedder import get_model, encode_texts, build_artist_text

# Load the model (downloads ~90 MB on first run)
model = get_model()
print('Model loaded:', model)
print('Embedding dimension:', model.get_sentence_embedding_dimension())

In [ ]:
# Sample music-related descriptions (analogous to movie overviews in the lab)
sample_texts = [
    "Artist: Radiohead. Experimental alternative rock band from Oxford known for complex soundscapes.",
    "Artist: Thom Yorke. Frontman of Radiohead, solo experimental electronic and art rock artist.",
    "Artist: Coldplay. British pop rock band with anthemic melodies and stadium-filling sound.",
    "Artist: Beyoncé. Global pop and R&B superstar with powerful vocals and dynamic performances.",
    "Artist: Kanye West. Influential hip-hop producer and rapper who shaped modern rap culture.",
]

sample_embeddings = encode_texts(sample_texts)
print('Embedding shape:', sample_embeddings.shape)
print('First 8 values of embedding[0]:', sample_embeddings[0][:8].round(4))

**Observation:** The output shape `(5, 384)` confirms that five texts were converted into embeddings and each embedding contains 384 numerical values. The vectors are unit-normalised (L2 norm ≈ 1.0), which means cosine similarity equals the dot product.

In [ ]:
# Verify normalisation: all L2 norms should be ~1.0
norms = np.linalg.norm(sample_embeddings, axis=1)
print('L2 norms (should all be ~1.0):', norms.round(5))

---
## Part 3 — Computing Similarity Between Texts

| Measure | Range | Interpretation |
|---|---|---|
| Cosine similarity | −1 to 1 | 1 = identical, 0 = unrelated, −1 = opposite |
| Dot product | −∞ to +∞ | Equals cosine when vectors are normalised |
| Euclidean distance | 0 to +∞ | 0 = identical, larger = more different |

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim_matrix = cosine_similarity(sample_embeddings)

labels = ['Radiohead', 'Thom Yorke', 'Coldplay', 'Beyoncé', 'Kanye West']
cos_df = pd.DataFrame(cos_sim_matrix, index=labels, columns=labels)

print('Cosine Similarity Matrix:')
print(cos_df.round(4).to_string())

In [ ]:
# Visualise the similarity matrix as a heatmap
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cos_sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_yticklabels(labels)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{cos_sim_matrix[i, j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Sample Artist Embedding Cosine Similarity')
plt.tight_layout()
plt.show()

**Observation:** Radiohead and Thom Yorke have the highest similarity score because both descriptions mention experimental and alternative rock. Coldplay, Beyoncé, and Kanye West are less similar to each other despite all being popular artists.

In [ ]:
# Semantic search over the 5 sample texts
query = "experimental alternative rock from England"
query_emb = encode_texts([query])  # shape (1, 384)

similarities = cosine_similarity(query_emb, sample_embeddings)[0]

ranked = sorted(zip(labels, similarities), key=lambda x: -x[1])
print(f'Query: "{query}"\n')
print(f'{"Rank":<6}{"Artist":<15}{"Cosine Similarity"}')
print('-' * 40)
for rank, (name, score) in enumerate(ranked, 1):
    print(f'{rank:<6}{name:<15}{score:.4f}')

**Observation:** Radiohead ranks first and Thom Yorke second — both match the query about experimental rock. The query does not use the word "Radiohead" but the model still finds them because it understands *meaning*, not just exact words.

In [ ]:
# Compare all three similarity measures on two specific pairs
# Pair A: Radiohead vs Thom Yorke  (similar)
# Pair B: Radiohead vs Beyoncé     (dissimilar)

def all_similarities(emb_a: np.ndarray, emb_b: np.ndarray):
    cos  = float(np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b)))
    dot  = float(np.dot(emb_a, emb_b))
    eucl = float(np.linalg.norm(emb_a - emb_b))
    return cos, dot, eucl

pair_a = all_similarities(sample_embeddings[0], sample_embeddings[1])
pair_b = all_similarities(sample_embeddings[0], sample_embeddings[3])

comparison = pd.DataFrame({
    'Pair': ['Radiohead vs Thom Yorke (similar)', 'Radiohead vs Beyoncé (dissimilar)'],
    'Cosine Similarity': [pair_a[0], pair_b[0]],
    'Dot Product':       [pair_a[1], pair_b[1]],
    'Euclidean Distance':[pair_a[2], pair_b[2]],
})
print(comparison.round(4).to_string(index=False))

**Observation:** For the similar pair, cosine similarity and dot product are *higher* and Euclidean distance is *lower*. For the dissimilar pair the pattern reverses. Since vectors are unit-normalised, cosine similarity equals the dot product exactly.

---
## Part 4 — Setting Up ChromaDB

ChromaDB is a vector database that persists embeddings to disk so they only need to be generated once.

| ChromaDB Concept | Music Pipeline Equivalent |
|---|---|
| Client | Connection to the persistent database |
| Collection | A table holding artists with embeddings and metadata |
| Document | The text string embedded (artist name + popularity info) |
| Embedding | The 384-number vector from `all-MiniLM-L6-v2` |
| Metadata | name, listeners, playcount, source_collection |
| ID | Unique string: `artist_<index>` |

In [ ]:
from embeddings.chroma_store import get_client, get_or_create_collection, COLLECTION_NAME, DEFAULT_DB_PATH

print('ChromaDB path:', DEFAULT_DB_PATH)
print('Collection name:', COLLECTION_NAME)

client = get_client()
collection = get_or_create_collection(client, reset=False)

print(f'Collection "{collection.name}" ready. Current count: {collection.count()}')
print('Similarity metric: cosine (set via hnsw:space metadata)')

---
## Part 5 — Adding Artists to ChromaDB

We load the cleaned dataset and embed every artist. ChromaDB persists the vectors to `data/embeddings/chroma_db/` so subsequent runs skip already-stored artists.

In [ ]:
from embeddings.chroma_store import add_artists

CLEAN_CSV = ROOT / 'data' / 'processed' / 'cleaned' / 'clean.csv'
df = pd.read_csv(CLEAN_CSV)

print('Cleaned dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

In [ ]:
# Show what text each artist row becomes before embedding
print('Example embedded texts:')
for _, row in df.head(4).iterrows():
    text = build_artist_text(row)
    print(f'  -> {text}')

In [ ]:
# Populate ChromaDB (first run embeds all artists; subsequent runs skip existing ones)
added = add_artists(collection, df, skip_existing=True)
print(f'Artists added this run: {added}')
print(f'Total in collection:    {collection.count()}')

---
## Part 6 — Querying ChromaDB by Text Similarity

ChromaDB automatically embeds the query and returns the closest matches by cosine distance.  
A single API call handles multiple queries in one round-trip.

In [ ]:
from embeddings.chroma_store import query_collection

# Multi-query call in a single API call
queries = [
    "popular rock band with millions of listeners",
    "hip hop and rap music",
]

all_results = query_collection(collection, queries, n_results=4)

for q, results in zip(queries, all_results):
    print(f'\nQuery: "{q}"')
    print(f'{"Rank":<5}{"Artist":<20}{"Similarity":<12}{"Listeners"}')
    print('-' * 55)
    for i, r in enumerate(results, 1):
        name = r['metadata'].get('name', r['id'])
        sim  = r['similarity']
        listeners = r['metadata'].get('listeners', 'N/A')
        l_str = f"{int(listeners):,}" if isinstance(listeners, (int, float)) else 'N/A'
        print(f'{i:<5}{name:<20}{sim:<12.4f}{l_str}')

**Observation:** Both queries return different artists ranked by semantic similarity. ChromaDB embeds the query text and performs approximate nearest-neighbour search, finding meaningful matches even when exact words differ.

---
## Part 7 — Metadata Filtering

ChromaDB supports combining semantic search with structured metadata filters.

| Operator | Meaning |
|---|---|
| `$eq` | Equal to |
| `$ne` | Not equal to |
| `$gt` / `$gte` | Greater than / ≥ |
| `$lt` / `$lte` | Less than / ≤ |
| `$in` | Value in a list |
| `$nin` | Value NOT in a list |
| `$and` | All conditions must be true |
| `$or` | At least one condition must be true |

In [ ]:
def show_results(label, results):
    print(f'\n--- {label} ---')
    if not results:
        print('  (no results)')
        return
    for i, r in enumerate(results, 1):
        name = r['metadata'].get('name', r['id'])
        sim  = r['similarity']
        src  = r['metadata'].get('source_collection', '?')
        listeners = r['metadata'].get('listeners', None)
        l_str = f"{int(listeners):,}" if isinstance(listeners, (int, float)) else 'N/A'
        print(f'  {i}. {name:<20} sim={sim:.3f}  src={src:<18}  listeners={l_str}')

q = "famous music artist with many fans"

In [ ]:
# Filter 1: $eq — only artists from the lastfm_api source
results_eq = query_collection(
    collection, [q], n_results=4,
    where={"source_collection": {"$eq": "lastfm_api"}}
)[0]
show_results('$eq: source_collection == lastfm_api', results_eq)

In [ ]:
# Filter 2: $gte — only artists with >= 1,000,000 listeners
results_gte = query_collection(
    collection, [q], n_results=4,
    where={"listeners": {"$gte": 1_000_000}}
)[0]
show_results('$gte: listeners >= 1,000,000', results_gte)

In [ ]:
# Filter 3: $in — artists from either lastfm_api or lastfm_json
results_in = query_collection(
    collection, [q], n_results=4,
    where={"source_collection": {"$in": ["lastfm_api", "lastfm_json"]}}
)[0]
show_results('$in: source in [lastfm_api, lastfm_json]', results_in)

In [ ]:
# Filter 4: $and — source == lastfm_api AND listeners >= 2,000,000
results_and = query_collection(
    collection, [q], n_results=4,
    where={
        "$and": [
            {"source_collection": {"$eq": "lastfm_api"}},
            {"listeners": {"$gte": 2_000_000}},
        ]
    }
)[0]
show_results('$and: source==lastfm_api AND listeners>=2M', results_and)

In [ ]:
# Filter 5: $or — listeners > 5,000,000 OR source == scraped_web_data
results_or = query_collection(
    collection, [q], n_results=4,
    where={
        "$or": [
            {"listeners": {"$gt": 5_000_000}},
            {"source_collection": {"$eq": "scraped_web_data"}},
        ]
    }
)[0]
show_results('$or: listeners>5M OR source==scraped_web_data', results_or)

In [ ]:
# Filter 6: $ne — exclude scraped_web_data sources
results_ne = query_collection(
    collection, [q], n_results=4,
    where={"source_collection": {"$ne": "scraped_web_data"}}
)[0]
show_results('$ne: source != scraped_web_data', results_ne)

**Observation:** Metadata filters let us narrow semantic search results without re-running the embedding step. We combined meaning-based retrieval with structured constraints — this is something keyword search alone cannot do efficiently.

---
## Part 8 — Search Engine Module

The `search_engine.py` module provides three clean, reusable functions:
- `semantic_search()` — ChromaDB meaning-based retrieval
- `keyword_search()` — exact substring matching over DataFrame columns
- `compare_search()` — runs both and prints side-by-side

In [ ]:
from embeddings.search_engine import semantic_search, keyword_search, compare_search

# Semantic search
sem_results = semantic_search("electronic dance music DJ", n_results=5)
print('Semantic search: "electronic dance music DJ"')
for i, r in enumerate(sem_results, 1):
    name = r['metadata'].get('name', r['id'])
    print(f'  {i}. {name}  (similarity={r["similarity"]:.4f})')

In [ ]:
# Keyword search
kw_results = keyword_search("Pink", df, n_results=5)
print('Keyword search: "Pink"')
for i, r in enumerate(kw_results, 1):
    print(f'  {i}. {r["name"]}  (matched in col: {r["matched_column"]})')

In [ ]:
# Side-by-side comparison
results = compare_search("rap and hip hop", df, n_results=5)

---
## Part 9 — Complete Semantic Search System

We now connect all components and run a multi-query demo.

In [ ]:
demo_queries = [
    "indie alternative rock",
    "R&B soul singer",
    "punk and heavy metal",
    "classical orchestral music",
]

for query in demo_queries:
    compare_search(query, df, n_results=4)

### When does each approach work better?

**Semantic search works better when:**
- You search with synonyms or paraphrases ("rapper" vs "hip hop artist" vs "MC")
- You describe the *style* rather than the *name* ("melancholic guitar-driven rock")
- The query and the stored text use different vocabulary for the same concept
- You want exploratory discovery across the collection

**Keyword search works better when:**
- You know the exact artist name ("Radiohead", "PinkPantheress")
- Precision is critical and false positives are costly
- The query is an identifier, ID, or code where meaning does not apply
- The text data is highly structured and terminology is standardised

---
## Part 10 — Keyword vs Semantic Search Comparison

We use synonym query pairs to measure result overlap. Higher overlap = the method handles synonyms consistently.

In [ ]:
synonym_pairs = [
    ("pop music",            "mainstream popular songs"),
    ("rock band",            "guitar-driven alternative group"),
    ("hip hop",              "rap music"),
    ("electronic music",     "EDM synth beats"),
]

N = 8  # results per query

rows = []
for q1, q2 in synonym_pairs:
    kw1 = set(r['name'] for r in keyword_search(q1, df, n_results=N))
    kw2 = set(r['name'] for r in keyword_search(q2, df, n_results=N))
    kw_overlap = len(kw1 & kw2)

    sem1 = set(r['metadata'].get('name', r['id']) for r in semantic_search(q1, n_results=N))
    sem2 = set(r['metadata'].get('name', r['id']) for r in semantic_search(q2, n_results=N))
    sem_overlap = len(sem1 & sem2)

    rows.append({
        'Query A': q1,
        'Query B': q2,
        'Keyword Overlap': kw_overlap,
        'Semantic Overlap': sem_overlap,
    })

overlap_df = pd.DataFrame(rows)
print(overlap_df.to_string(index=False))

In [ ]:
# Bar chart comparing overlap
x = range(len(synonym_pairs))
width = 0.35
labels_short = [f'Pair {i+1}' for i in x]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - width/2 for i in x], overlap_df['Keyword Overlap'],  width, label='Keyword', color='steelblue')
ax.bar([i + width/2 for i in x], overlap_df['Semantic Overlap'], width, label='Semantic', color='darkorange')
ax.set_xticks(list(x))
ax.set_xticklabels([f'{q1[:20]}...\nvs {q2[:20]}...' for q1, q2 in synonym_pairs], fontsize=8)
ax.set_ylabel('Result Overlap (out of 8)')
ax.set_title('Keyword vs Semantic Search — Synonym Overlap')
ax.legend()
ax.set_ylim(0, N + 1)
plt.tight_layout()
plt.show()

**Observation:** Semantic search consistently produces higher overlap between synonym query pairs compared to keyword search. This confirms that semantic search handles paraphrases and vocabulary variation much better than exact word matching.

---
## Part 11 — Hybrid Search (Reciprocal Rank Fusion)

**Reciprocal Rank Fusion (RRF):**
```
score(d) = Σ  1 / (k + rank(d))
```
where `k = 60` prevents the top result from dominating.  
Results that appear high in *both* lists get the highest combined score.

In [ ]:
from embeddings.hybrid_search import hybrid_search, print_hybrid_comparison

hyb_results = hybrid_search("popular band", df, n_results=6, k=60)

print('Hybrid Search Results — "popular band"')
print(f'{"Rank":<5}{"Artist":<22}{"RRF Score":<12}{"KW Rank":<10}{"Sem Rank":<10}{"Listeners"}')
print('-' * 70)
for i, r in enumerate(hyb_results, 1):
    l = r['listeners']
    l_str = f"{int(l):,}" if isinstance(l, (int, float)) else 'N/A'
    kw_r = str(r['keyword_rank'])  if r['keyword_rank']  else '-'
    se_r = str(r['semantic_rank']) if r['semantic_rank'] else '-'
    print(f"{i:<5}{r['name'][:20]:<22}{r['rrf_score']:<12.6f}{kw_r:<10}{se_r:<10}{l_str}")

In [ ]:
# Three-way comparison for multiple queries
for query in ["rock legend", "electronic producer"]:
    print_hybrid_comparison(query, df, n_results=5)

**Observation:** Hybrid search combines the precision of keyword matching with the recall of semantic search. Artists that appear in both ranked lists are promoted to the top. Artists only found by one method still appear but with lower combined scores.

---
## Part 12 — Pipeline Integration

The embedding stage has been added to `src/run_pipeline.py` as **Step 11**, after the cleaning stage.  
It calls `run_embedding_stage(cleaned_csv)` which:
1. Loads the cleaned CSV
2. Connects to the persistent ChromaDB collection
3. Embeds and adds any artists not yet in the database
4. Runs sample semantic queries and logs the results

The function is wrapped in a `try/except` block so the pipeline continues even if the embedding step fails.

In [ ]:
# Show the embedding stage function from run_pipeline.py
pipeline_path = ROOT / 'src' / 'run_pipeline.py'
lines = pipeline_path.read_text().splitlines()

in_func = False
for line in lines:
    if 'def run_embedding_stage' in line:
        in_func = True
    if in_func:
        print(line)
    if in_func and line.strip() == '' and 'def run_pipeline' in ''.join(lines[lines.index(line):lines.index(line)+3]):
        break

In [ ]:
# Verify the pipeline call site (step 11)
for i, line in enumerate(lines):
    if 'Embedding stage' in line or 'run_embedding_stage' in line:
        print(f'Line {i+1}: {line}')

---
## Part 13 — Analytical Questions

We use the full semantic search system to answer three real questions about the dataset.

### Question 1: Which artists in the dataset are most associated with "underground cult following" — artists beloved by a niche audience but not mainstream?

We combine a semantic query for "underground cult following" with a metadata filter for artists with fewer than 2,000,000 listeners.

In [ ]:
q1_results = semantic_search(
    "underground cult following niche dedicated fanbase",
    n_results=8,
    where={"listeners": {"$lt": 2_000_000}},
)

print('Q1: Underground/Niche Artists (< 2M listeners)')
print(f'{"Rank":<5}{"Artist":<25}{"Similarity":<12}{"Listeners"}')
print('-' * 55)
for i, r in enumerate(q1_results, 1):
    name = r['metadata'].get('name', r['id'])
    l = r['metadata'].get('listeners', None)
    l_str = f"{int(l):,}" if isinstance(l, (int, float)) else 'N/A'
    print(f"{i:<5}{name:<25}{r['similarity']:<12.4f}{l_str}")

**Analysis:** The results show artists with relatively fewer listeners but still present in the Last.fm top-chart data. The semantic filter for "underground cult following" combined with the `$lt` listener constraint surfaces less-mainstream artists. This kind of query is impossible with pure keyword search because no artist description explicitly says "underground" — the model infers it from context.

### Question 2: Who are the most globally dominant artists — those with massive reach and high play counts?

We search for "globally dominant superstar massive reach" and filter to artists with over 5 million listeners.

In [ ]:
q2_results = semantic_search(
    "globally dominant superstar massive fanbase worldwide reach",
    n_results=8,
    where={"listeners": {"$gte": 5_000_000}},
)

print('Q2: Global Superstars (>= 5M listeners)')
print(f'{"Rank":<5}{"Artist":<25}{"Similarity":<12}{"Listeners":<18}{"Playcount"}')
print('-' * 75)
for i, r in enumerate(q2_results, 1):
    name = r['metadata'].get('name', r['id'])
    l    = r['metadata'].get('listeners', None)
    p    = r['metadata'].get('playcount', None)
    l_str = f"{int(l):,}" if isinstance(l, (int, float)) else 'N/A'
    p_str = f"{int(p):,}" if isinstance(p, (int, float)) else 'N/A'
    print(f"{i:<5}{name:<25}{r['similarity']:<12.4f}{l_str:<18}{p_str}")

**Analysis:** This query returns artists who are genuinely globally popular, verified by the `$gte 5,000,000` listener filter. The semantic model captures the concept of "worldwide dominance" even though the stored artist texts only contain name and listener tier tags. The result reveals which big names in the Last.fm dataset have both high listener counts and high playcounts — indicating not just passive discovery but active repeat listening.

### Question 3: Does the scraped web data source produce different semantic search results compared to the API source for the same query?

We run the same semantic query against two different source filters and compare the results.

In [ ]:
q3_query = "iconic influential music legend"

api_results = semantic_search(
    q3_query, n_results=5,
    where={"source_collection": {"$eq": "lastfm_api"}},
)

scraped_results = semantic_search(
    q3_query, n_results=5,
    where={"source_collection": {"$eq": "scraped_web_data"}},
)

api_names     = [r['metadata'].get('name', r['id']) for r in api_results]
scraped_names = [r['metadata'].get('name', r['id']) for r in scraped_results]

print(f'Query: "{q3_query}"\n')
print(f'{"Rank":<5}{"API Source":<25}{"Scraped Web Source"}')
print('-' * 55)
for i in range(max(len(api_names), len(scraped_names))):
    a = api_names[i]     if i < len(api_names)     else ''
    s = scraped_names[i] if i < len(scraped_names) else ''
    print(f'{i+1:<5}{a:<25}{s}')

overlap = set(api_names) & set(scraped_names)
print(f'\nOverlap: {overlap if overlap else "(none — different artists in each source)"}')

**Analysis:** This question reveals whether the two data ingestion sources (API pull vs web scraping) captured overlapping or distinct artists. If there is no overlap, it means the scraper captured a different subset of artists than the API — which is expected since scraped pages may represent curated lists while the API returns top-chart data. The metadata filter makes this comparison trivial in semantic search, whereas keyword search would require manual DataFrame filtering followed by a separate text scan.

---
## Summary

| Component | Location | Purpose |
|---|---|---|
| `embedder.py` | `src/embeddings/` | Load `all-MiniLM-L6-v2`, encode texts, build artist text |
| `chroma_store.py` | `src/embeddings/` | Persistent ChromaDB collection, add/query artists |
| `search_engine.py` | `src/embeddings/` | `semantic_search`, `keyword_search`, `compare_search` |
| `hybrid_search.py` | `src/embeddings/` | RRF fusion of keyword + semantic ranked lists |
| ChromaDB data | `data/embeddings/chroma_db/` | Persistent vector store (survives kernel restart) |
| Pipeline step | `src/run_pipeline.py` step 11 | Automatic embedding after cleaning |

**Key takeaways:**
- Embeddings capture semantic meaning — the model finds "rap" when you search "hip hop" even with no text overlap
- ChromaDB's persistent storage means embeddings are generated once, not every run
- Metadata filters let you combine meaning-based search with structured constraints (listener counts, sources)
- Hybrid search (RRF) outperforms either method alone: it keeps keyword precision and adds semantic recall